# 01. CLIP-Catalog: Data Exploration & Zero-Shot Baseline

This notebook demonstrates:
1. **Catalog Dataset Ingestion & Schema Validation** (Quick-commerce SKU metadata + imagery)
2. **OpenCLIP ViT-B/32 Zero-Shot Baseline** (Evaluating out-of-the-box accuracy before fine-tuning)
3. **Cross-Modal Cosine Similarity Heatmaps** (Image embeddings vs text category prompt embeddings)
4. **Threshold Calibration for OnboardingBridge** (`auto_tag`, `human_review`, `reject` boundary inspection)

In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd

# Ensure project root is accessible
PROJECT_ROOT = os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.data import generate_synthetic_catalog
from src.onboarding_bridge import OnboardingBridge
from src.eval import compute_retrieval_metrics

print("Project root configured:", PROJECT_ROOT)

## 1. Load or Generate Catalog Samples

In [ ]:
# Generate a representative quick-commerce catalog sample (100 SKUs)
catalog_items = generate_synthetic_catalog(num_skus=100)
df_catalog = pd.DataFrame(catalog_items)

print(f"Loaded {len(df_catalog)} catalog entries.")
display(df_catalog[['sku_id', 'title', 'brand', 'category', 'subcategory']].head(10))

print("\nCategory distribution:")
print(df_catalog['category'].value_counts())

## 2. Simulate Contrastive Embeddings & Cosine Similarity

We compute the cross-modal similarity matrix $S = X_{\text{image}} \cdot Y_{\text{text}}^T$ where both embedding sets are $\ell_2$-normalized.

In [ ]:
np.random.seed(42)
dim = 512
categories = sorted(df_catalog['category'].unique())
cat_to_idx = {cat: i for i, cat in enumerate(categories)}

# Construct prototype text embeddings for each category
text_embeds = np.random.randn(len(categories), dim)
text_embeds /= np.linalg.norm(text_embeds, axis=-1, keepdims=True)

# Construct synthetic image embeddings correlated with true category
img_embeds = []
for item in catalog_items:
    true_idx = cat_to_idx[item['category']]
    # Correlate with true text prototype plus noise (simulating zero-shot baseline)
    vec = 0.65 * text_embeds[true_idx] + 0.35 * np.random.randn(dim)
    vec /= np.linalg.norm(vec)
    img_embeds.append(vec)

img_embeds = np.array(img_embeds)

# Compute cosine similarities: shape (num_skus, num_categories)
similarity_matrix = np.dot(img_embeds, text_embeds.T)
print("Similarity matrix shape:", similarity_matrix.shape)

## 3. Zero-Shot Softmax Probabilities & Top-1 Accuracy

In [ ]:
temperature = 0.07
scaled_logits = similarity_matrix / temperature
exp_logits = np.exp(scaled_logits - np.max(scaled_logits, axis=-1, keepdims=True))
probs = exp_logits / np.sum(exp_logits, axis=-1, keepdims=True)

pred_indices = np.argmax(probs, axis=-1)
true_indices = np.array([cat_to_idx[item['category']] for item in catalog_items])

top1_acc = np.mean(pred_indices == true_indices)
print(f"Zero-Shot Baseline Top-1 Accuracy: {top1_acc * 100:.2f}%")

## 4. OnboardingBridge Commercial Decisions

Testing autonomous decision gating (`auto_tag`, `human_review`, `reject`) on top predictions.

In [ ]:
bridge = OnboardingBridge(
    auto_tag_confidence=0.70,
    review_queue_confidence=0.40,
    min_top1_margin=0.15
)

actions = []
for i, item in enumerate(catalog_items):
    scores_dict = {cat: float(probs[i, j]) for j, cat in enumerate(categories)}
    decision = bridge.decide(scores_dict, item)
    actions.append(decision['action'])

action_counts = pd.Series(actions).value_counts()
print("OnboardingBridge Decisions Distribution:")
print(action_counts)
print(f"\nPercentage auto-tagged without manual intervention: {action_counts.get('auto_tag', 0) / len(catalog_items) * 100:.1f}%")